# The End-to-End Data Science Pipeline

**Topic:** Orientation

Every data science project follows the same underlying structure. The tools and techniques change. The pipeline doesn't. This notebook shows you the complete picture before we zoom in on any of the details.

---
## The big picture

The pipeline has nine stages. Each one builds on the previous. Use the dropdown below to explore what each stage does, what question it answers, where to find it in this curriculum, and the most common mistake made there.

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, Output, VBox
from IPython.display import display, clear_output, Markdown
from tkh_utils import PALETTE, FONT, base_layout

STAGES = [
    {
        "name": "1. Define Problem",
        "description": "Translate a business or research question into something a data scientist can work with. What are we trying to predict or understand? What does success look like?",
        "question": "What problem are we actually solving, and how will we know if we solved it?",
        "folder": "This orientation — and again in every project and case study.",
        "mistake": "Skipping this stage entirely. Jumping straight to data collection without a clear problem statement leads to aimless analysis.",
    },
    {
        "name": "2. Collect Data",
        "description": "Gather the raw data needed to answer the question. This might mean downloading a dataset, querying a database, scraping a website, or running a survey.",
        "question": "What data do we need, where does it live, and how do we access it?",
        "folder": "eda/ covers data loading; tools/ covers APIs and SQL basics.",
        "mistake": "Assuming the data you have is the data you need. The data you have is often a proxy — understand what it actually represents.",
    },
    {
        "name": "3. EDA",
        "description": "Exploratory Data Analysis. Before modeling anything, look at the data from every angle. Plot distributions. Check for missing values. Find outliers. Ask: does this make sense?",
        "question": "What does the data actually look like, and what does it tell us before we do anything to it?",
        "folder": "eda/ — an entire folder dedicated to this stage.",
        "mistake": "Treating EDA as a checkbox rather than a discovery process. Most project failures trace back to skipping real exploration.",
    },
    {
        "name": "4. Preprocess",
        "description": "Clean and prepare the data for modeling. Handle missing values, remove or cap outliers, encode categorical variables, and scale numerical features.",
        "question": "Is the data in a form where a machine learning algorithm can actually learn from it?",
        "folder": "preprocessing/ — covers every technique in depth.",
        "mistake": "Applying the same transformations to training and test data incorrectly (data leakage). This is one of the most common sources of misleading model performance.",
    },
    {
        "name": "5. Engineer Features",
        "description": "Create new variables from existing ones that better capture the underlying patterns. A timestamp becomes day-of-week. Two columns become a ratio. Domain knowledge lives here.",
        "question": "What additional signals can we extract from the data that the model does not already have?",
        "folder": "feature_engineering/ — covered in its own folder.",
        "mistake": "Feature engineering after seeing test set results. Features must be designed from domain knowledge, not from what makes the test score go up.",
    },
    {
        "name": "6. Model",
        "description": "Select and train a machine learning algorithm on the prepared data. The algorithm adjusts its internal parameters until it finds patterns that generalize to new examples.",
        "question": "Which algorithm fits this problem, and what did it learn from the data?",
        "folder": "supervised/, unsupervised/, ml_concepts/",
        "mistake": "Jumping straight to the most complex model. Start simple. A linear model that you understand beats a neural network you cannot explain.",
    },
    {
        "name": "7. Evaluate",
        "description": "Measure how well the model performs — on data it has never seen before. Choose the right metric for the problem. Understand what the score actually means.",
        "question": "Is this model good enough to use, and good enough for what?",
        "folder": "model_evaluation/ — an entire folder on this stage.",
        "mistake": "Evaluating on training data. A model that scores 100% on training data has almost certainly memorized the data rather than learned from it.",
    },
    {
        "name": "8. Deploy",
        "description": "Make the model available for others to use — as an API, a dashboard, a scheduled batch job, or an embedded feature in a product.",
        "question": "How do other people or systems get predictions from this model?",
        "folder": "Covered conceptually in case_studies/. Full deployment engineering is beyond this curriculum.",
        "mistake": "Building a model that only runs on your laptop. If it cannot be used by others, it cannot create value.",
    },
    {
        "name": "9. Monitor",
        "description": "Watch the model performance over time. The world changes — customer behavior shifts, data pipelines break, distributions drift. A model that worked last year may not work today.",
        "question": "Is the model still performing as expected, and what do we do when it is not?",
        "folder": "Covered conceptually in case_studies/ and model_evaluation/.",
        "mistake": "Treating deployment as the finish line. Models degrade in production. Monitoring is not optional — it is part of the project.",
    },
]

out = Output()

stage_dropdown = Dropdown(
    options=[s["name"] for s in STAGES],
    value=STAGES[0]["name"],
    description="Select stage:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"),
)

def render(stage_name):
    stage = next(s for s in STAGES if s["name"] == stage_name)
    idx = STAGES.index(stage)
    colors = [PALETTE["muted"]] * len(STAGES)
    colors[idx] = PALETTE["primary"]

    fig = go.Figure()
    for i, s in enumerate(STAGES):
        fig.add_trace(go.Bar(
            x=[1], y=[s["name"]],
            orientation="h",
            marker_color=colors[i],
            showlegend=False,
            hoverinfo="skip",
            width=0.6,
        ))

    layout = base_layout(title="Pipeline Stage: " + stage["name"])
    layout.update(
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[0, 1.5]),
        yaxis=dict(autorange="reversed"),
        height=340,
        margin=dict(l=175, r=20, t=60, b=20),
    )
    fig.layout = layout

    info_lines = (
        "**What it does:** " + stage["description"] + "\n\n"
        + "**Key question:** *" + stage["question"] + "*\n\n"
        + "**Where in the curriculum:** " + stage["folder"] + "\n\n"
        + "**Common mistake:** " + stage["mistake"]
    )

    with out:
        clear_output(wait=True)
        fig.show()
        display(Markdown(info_lines))

stage_dropdown.observe(lambda change: render(change["new"]), names="value")
display(VBox([stage_dropdown, out]))
render(STAGES[0]["name"])

---
## A complete example: Predicting house prices

The best way to understand the pipeline is to walk through it once, end to end. We will use a real dataset — the California Housing dataset — and touch every stage. The goal is not to build the best possible model. The goal is to see how all the stages connect.

---

### Stage 1 — Define the problem

**The question:** Can we predict median house prices in California neighborhoods from characteristics like location, age, and income level?

**Why this matters:** A real estate platform could use this to give buyers a price estimate before they visit a property. A bank could use it to flag mortgage applications where the sale price looks unusual.

**What success looks like:** A model that predicts prices close enough to actual values to be useful — not perfect, but better than guessing the average.

In [2]:
# Stage 2 — Collect and inspect the data
from sklearn.datasets import fetch_california_housing
import pandas as pd

housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


The dataset has 20,640 rows — one per census block group in California — and these features:

| Feature | Plain English |
|---------|---------------|
| `MedInc` | Median household income in the block group |
| `HouseAge` | Median age of houses in the block group |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Total population of the block group |
| `AveOccup` | Average number of people per household |
| `Latitude` | Geographic latitude |
| `Longitude` | Geographic longitude |
| `MedHouseVal` | **Target:** Median house value (in $100,000s) |

In [3]:
# Stage 3 — Exploratory data analysis
import plotly.graph_objects as go
from tkh_utils import PALETTE, FONT, base_layout

layout = base_layout(
    title="Distribution of Median House Values (California Housing Dataset)",
    xaxis_title="Median House Value ($100,000s)",
    yaxis_title="Count"
)
layout.update(height=380, margin=dict(l=60, r=30, t=70, b=60))

fig = go.Figure(data=[
    go.Histogram(
        x=df["MedHouseVal"],
        nbinsx=50,
        marker_color=PALETTE["primary"],
        opacity=0.8,
    )
], layout=layout)
fig.show()

**One insight from the distribution:** There is a hard spike at 5.0 — the dataset caps values at $500,000. This is a data artifact, not a real pattern. A real data scientist would flag this before modeling and decide how to handle it. We note it and move on for this walkthrough.

In [4]:
# Stage 4 — Preprocess
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape[0]} rows | Test set: {X_test_scaled.shape[0]} rows")
print(f"After scaling — Training mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")

Training set: 16512 rows | Test set: 4128 rows
After scaling — Training mean: 0.0000, std: 1.0000


**Why scaling?** Many machine learning algorithms treat features with large numeric ranges as more important than features with small ranges, even when that is not true. Scaling brings all features onto the same scale — usually with mean 0 and standard deviation 1 — so the algorithm can compare them fairly.

We fit the scaler on training data only, then apply it to both sets. This prevents the test set from "leaking" information into the training process.

In [5]:
# Stage 5 — Feature engineering
import pandas as pd

def add_features(X_df):
    X_df = X_df.copy()
    X_df["RoomsPerPerson"] = X_df["AveRooms"] / X_df["AveOccup"].clip(lower=0.5)
    return X_df

X_train_feat = add_features(X_train)
X_test_feat  = add_features(X_test)

scaler2 = StandardScaler()
X_train_scaled2 = scaler2.fit_transform(X_train_feat)
X_test_scaled2  = scaler2.transform(X_test_feat)

print("Features after engineering:", list(X_train_feat.columns))

Features after engineering: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'RoomsPerPerson']


**Why this feature?** Average rooms per person is more meaningful than raw room count. A house with 8 rooms occupied by 8 people is very different from the same house with 1 person. Domain knowledge tells us that density matters for price.

In [6]:
# Stage 6 — Model
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_scaled2, y_train)

print("The model learned from the data.")
print(f"It has {len(model.coef_)} coefficients — one per feature.")
print(f"Intercept (baseline prediction): {model.intercept_:.3f}")

The model learned from the data.
It has 9 coefficients — one per feature.
Intercept (baseline prediction): 2.072


In [7]:
# Stage 7 — Evaluate
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test_scaled2)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
print(f"R\u00b2 Score:                    {r2:.3f}")
print()
print(f"In plain English: our predictions are off by about ${rmse * 100_000:,.0f} on average.")
print(f"The model explains {r2 * 100:.1f}% of the variation in house prices.")

Root Mean Squared Error (RMSE): 0.687
R² Score:                    0.640

In plain English: our predictions are off by about $68,731 on average.
The model explains 64.0% of the variation in house prices.


**What these scores mean:**

An RMSE near 0.73 means our predictions are off by about $73,000 on average. For a linear model with minimal feature engineering, that is a reasonable starting point — not a finished product, but proof that the pipeline works.

An R² near 0.60 means our model explains about 60% of the variation in house prices. The remaining 40% is variation the model cannot capture — some of which might be explained by features we do not have, like school quality or neighborhood safety.

A real project would now iterate: try different models, engineer more features, tune hyperparameters. But the structure of that iteration follows exactly this same pipeline.

---
### Stage 8 — Deploy (conceptual)

Deployment means making the model available to others. In a real project, you might wrap the model in a web API — so that when someone enters a neighborhood's characteristics into a website, the model returns a predicted price in real time. You might also deploy it as a batch job that runs every night and updates a table in a database.

Deployment engineering is its own discipline and is beyond the scope of this curriculum. But the concept matters: a model that only runs on your laptop has no value until someone else can use it.

---

### Stage 9 — Monitor (conceptual)

House prices change. The relationships between features and prices shift over time as neighborhoods evolve, the economy moves, and new housing stock appears. A model trained on 2015 data may be badly wrong by 2025.

Monitoring means regularly checking whether the model's predictions still match reality. When they stop matching, you retrain — going back to Stage 2 or Stage 4 and repeating the pipeline with fresh data.

The pipeline is not a one-time process. It is a cycle.

---
## Why the order matters

The stages build on each other in ways that are not optional.

You cannot preprocess before you have explored the data. How do you handle missing values if you do not know which columns have them, and in what pattern? How do you know which outliers to cap if you have not looked at the distributions first?

You cannot model before you have preprocessed. Most algorithms assume the data is in a certain form — scaled, encoded, complete. Feed raw data to a gradient boosting model and you will either get an error or silently wrong results.

You cannot evaluate fairly if you have seen the test set during training. The test set is supposed to represent data the model has never encountered. The moment you use it to make any decision — which features to include, which model to choose — it becomes training data, and your evaluation is no longer trustworthy.

The stages exist in this order because each one depends on what the previous one produced.

---
## The pipeline is never linear

Here is the honest version: real data science projects loop back constantly.

You do EDA and discover that 30% of a key column is missing. You go back to data collection. You train a model and realize the predictions are nonsensical in ways that suggest a preprocessing error. You go back to Stage 4. You show your results to a domain expert and they tell you the feature you engineered does not make business sense. You go back to Stage 5.

This iteration is not failure. It is the process. The pipeline is a mental model for organizing the work, not a rigid sequence that every project follows step by step.

> **Every data science project follows this pipeline. The tools and techniques change. The pipeline doesn't.**

---

## Before you move on

Sit with these questions:

1. Looking at the nine stages, which one do you think is most often skipped or rushed in real projects? Why?
2. The worked example used Linear Regression, the simplest possible model. What would you need to know to decide whether to use a more complex model instead?
3. Think about the "pipeline is never linear" idea. Does that feel freeing or unsettling to you, and why?

---

*Next up: How to navigate this curriculum* → `05_how_to_use_this_curriculum.ipynb`